# Phase 3: GUI-grounding LoRA training on Kaggle (sessions that survive)

Qwen2-VL-2B + LoRA on the v3 native-Windows grounding dataset (1572 train / 555 dev).
The full schedule is 3 epochs = **1179 optimizer steps**, more than one 12-hour Kaggle session holds,
so the run is split across sessions **by wall-clock time**, and every checkpoint is pushed to a
**private Hugging Face repo while training runs**. A session that dies loses at most ~30 minutes.

Why this notebook replaces `phase3_lora_training.ipynb`:
1. That notebook stopped each chunk with `max_steps = current + 400`. The learning-rate schedule is
   built from `max_steps`, so every chunk decayed the LR to zero at its own end. Here the schedule is
   always the full 1179 steps and a session ends on time instead.
2. Its checkpoints only survived if the whole notebook finished. Here they leave the machine as they are made.

## One-time setup
1. **huggingface.co** -> Settings -> Access Tokens -> create a token with **write** access.
2. In this notebook: **Add-ons -> Secrets -> Add secret**, label `HF_TOKEN`, paste the token, and tick it for this notebook.
   (Never paste the token into a cell.)
3. Right panel -> **Session options**: Accelerator **GPU T4 x2**, Internet **On**.
4. Right panel -> **Input**: attach the v3 dataset, and nothing else that contains a `labels.jsonl`.

## How to run (Save Version -> "Save & Run All (Commit)" every time, then you can close the laptop)
| # | `RUN_MODE` | What it proves | Expected log |
|---|---|---|---|
| 1 | `"smoke"` | secret works in a commit, Hub writes work, memory fits, real s/step | `pushed step 3/6`, `GPU peak ... GB`, ends `SMOKE PART 1 OK` |
| 2 | `"smoke"` | resume works on Kaggle for real | `resuming from step 3`, ends `SMOKE PASSED` |
| 3 | `"train"` | the real run, first ~11 h | `[clock]` progress lines, `[hub] pushed step ...` every ~30 min |
| 4+ | `"train"` | continue | `resuming from step N`, until `TRAINING FINISHED` |

The smoke run writes to a separate `...-smoke` repo, so it can never mix with the real run.

In [ ]:
# Cell 1 -- settings. Runs before ANY torch/transformers/huggingface_hub import, on purpose:
# pip installs pinned versions in cell 4, and a module imported before that would stay
# loaded in this kernel at Kaggle's preinstalled version.
import os
import time

SESSION_START = time.time()        # the session budget counts setup time too

RUN_MODE = "smoke"                 # "smoke" (commit it twice), then "train"
HF_REPO_NAME = "computer-use-grounder"
CODE_REF = "d1e135af1bbc31818fd869636fe31b6d980c2aa3"           # exact commit of rudranaresh0201/computer-use this run trains with
BUDGET_HOURS = 11.0                # Kaggle ends a session at 12 h; stop with an hour to spare
SAVE_EVERY_MINUTES = 30            # max work a dead session can cost

assert RUN_MODE in ("smoke", "train"), RUN_MODE

# One GPU: HF Trainer wraps 2 visible GPUs in DataParallel, which breaks Qwen2-VL's
# lazy `self.visual.dtype` (StopIteration) and doubles the effective batch size.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Lets the CUDA allocator grow segments instead of fragmenting -- the usual cause of an
# OOM hours into a run whose per-step memory is constant. (Both names: torch renamed it.)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(f"RUN_MODE={RUN_MODE}  CODE_REF={CODE_REF}  budget={BUDGET_HOURS} h")

In [ ]:
# Cell 2 -- the right GPU, checked without importing torch yet.
import subprocess

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0, "No GPU attached. Session options -> Accelerator -> GPU T4 x2."
assert "T4" in smi.stdout, (
    "Expected a Tesla T4. Session options -> Accelerator -> GPU T4 x2. "
    "(P100 has no tensor cores, so fp16 training would run several times slower.)"
)

In [ ]:
# Cell 3 -- fail in seconds, not hours: the token must be readable in THIS (committed) session
# and must be able to write. Plain HTTP, so huggingface_hub is not imported early.
import requests
from kaggle_secrets import UserSecretsClient

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Could not read the HF_TOKEN secret. Add-ons -> Secrets: add HF_TOKEN and tick it for this notebook."
    ) from e
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "HF_TOKEN secret is empty or not a Hugging Face token"

r = requests.get("https://huggingface.co/api/whoami-v2", headers={"Authorization": f"Bearer {HF_TOKEN}"}, timeout=30)
assert r.status_code == 200, f"Hugging Face rejected the token (HTTP {r.status_code}): {r.text[:200]}"
me = r.json()
role = me.get("auth", {}).get("accessToken", {}).get("role")
assert role != "read", "HF_TOKEN is a READ token. Create one with write access."
# Smoke runs get a fresh repo per code version, so a smoke test always starts at step 0
# on exactly the code it is testing. The real run keeps one repo across all its sessions.
HF_REPO = f"{me['name']}/{HF_REPO_NAME}" + (f"-smoke-{CODE_REF[:7]}" if RUN_MODE == "smoke" else "")
# Also used by the ~4 GB model download (unauthenticated downloads are rate-limited).
os.environ["HF_TOKEN"] = HF_TOKEN
print(f"token OK for {me['name']} (role: {role}); checkpoints go to private repo {HF_REPO}")

In [ ]:
# Cell 4 -- the exact code version, and its pinned dependencies.
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/computer-use")
if not REPO_DIR.is_dir():
    !git clone https://github.com/rudranaresh0201/computer-use.git {REPO_DIR}
!git -C {REPO_DIR} fetch --quiet origin
!git -C {REPO_DIR} checkout --quiet {CODE_REF}
!git -C {REPO_DIR} log -1 --format="training with commit %H (%s)"

# {sys.executable} -m pip, not bare pip: Kaggle can have several pythons on PATH.
!{sys.executable} -m pip install --quiet -e "{REPO_DIR}[training]"
# Kaggle preinstalls an old torchao that peft's LoRA dispatcher rejects; we never use it.
!{sys.executable} -m pip uninstall -y --quiet torchao

In [ ]:
# Cell 5 -- first heavy imports (now at the pinned versions), then a real write to the Hub.
import importlib
import sys

if str(REPO_DIR / "src") not in sys.path:  # editable install's .pth is only read at startup
    sys.path.insert(0, str(REPO_DIR / "src"))
importlib.invalidate_caches()

import huggingface_hub
import peft
import torch
import transformers

from computeruse.training.hub_checkpoint import (
    HubCheckpointCallback,
    HubSync,
    NanGuardCallback,
    WallClockCallback,
)

print(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__} | hub {huggingface_hub.__version__}")
assert transformers.__version__ == "5.0.0", "pinned transformers did not load -- restart and run from the top"
assert torch.cuda.is_available()
print(f"GPU: {torch.cuda.get_device_name(0)}, {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

sync = HubSync(HF_REPO, token=HF_TOKEN)
print(f"Hub preflight OK: wrote to {HF_REPO} as {sync.preflight()}")

In [ ]:
# Cell 6 -- the dataset: exactly one, and the version we expect.
from computeruse.training.dataset import resolve_path

candidates = sorted(Path("/kaggle/input").rglob("labels.jsonl"))
assert len(candidates) == 1, f"need exactly one labels.jsonl under /kaggle/input, found {candidates}"
DATASET_ROOT = candidates[0].parent


def count_rows(rel):
    return sum(1 for ln in resolve_path(DATASET_ROOT, rel).read_text(encoding="utf-8").splitlines() if ln.strip())


n_train, n_dev = count_rows("splits/train.jsonl"), count_rows("splits/dev.jsonl")
print(f"dataset {DATASET_ROOT}: train {n_train}, dev {n_dev}")
assert (n_train, n_dev) == (1572, 555), "not the v3 dataset -- detach the stale version"

In [ ]:
# Cell 7 -- where are we? Everything comes from the Hub; this container starts empty.
if RUN_MODE == "smoke":
    TOTAL_STEPS, SAVE_STEPS, EVAL_STEPS, EVAL_SUBSET = 6, 1000, 3, 8
else:
    TOTAL_STEPS, SAVE_STEPS, EVAL_STEPS, EVAL_SUBSET = 1179, 100, 100, 120  # 393 steps/epoch x 3

OUTPUT_DIR = Path("/kaggle/working/lora_grounder")
hub_state = sync.read_state()
print("Hub run_state.json:", hub_state)
if hub_state and hub_state["total_steps"] != TOTAL_STEPS:
    raise RuntimeError(
        f"The Hub run was planned for {hub_state['total_steps']} steps, this notebook plans {TOTAL_STEPS}. "
        "Resuming would mix two different learning-rate schedules. Use a new HF_REPO_NAME for a new run."
    )

ALREADY_FINISHED = bool(hub_state and hub_state["finished"])
resume_from = None if ALREADY_FINISHED else sync.fetch_last_checkpoint(Path("/kaggle/working/resume"))
START_STEP = hub_state["global_step"] if (hub_state and resume_from) else 0

if ALREADY_FINISHED:
    print(f"run already FINISHED at step {hub_state['global_step']} -- nothing to train")
elif resume_from:
    print(f"resuming from step {START_STEP}/{TOTAL_STEPS} ({resume_from})")
else:
    print(f"fresh start: step 0/{TOTAL_STEPS}")

# Smoke part 1 stops itself at step 3 so that part 2 has something to resume.
STOP_AFTER_STEP = 3 if (RUN_MODE == "smoke" and START_STEP == 0) else None

In [ ]:
# Cell 8 -- the trainer: full schedule every session, stops on the clock, pushes every save.
from computeruse.training.train_lora import build_trainer

clock_cb = WallClockCallback(
    session_start=SESSION_START,
    budget_seconds=BUDGET_HOURS * 3600,
    save_every_seconds=SAVE_EVERY_MINUTES * 60,
    stop_after_step=STOP_AFTER_STEP,
    progress_every_steps=1 if RUN_MODE == "smoke" else 10,
    gpu_peak_gb=lambda: torch.cuda.max_memory_allocated() / 1e9,
)
hub_cb = HubCheckpointCallback(sync, snapshot_every=100)
nan_cb = NanGuardCallback(patience=3)

if not ALREADY_FINISHED:
    trainer = build_trainer(
        DATASET_ROOT,
        OUTPUT_DIR,
        max_steps=TOTAL_STEPS if RUN_MODE == "smoke" else None,  # None = the 3-epoch schedule
        save_steps=SAVE_STEPS,
        eval_steps=EVAL_STEPS,
        load_best_model_at_end=False,  # best step is chosen later by dev click accuracy
        eval_subset_size=EVAL_SUBSET,
        callbacks=[clock_cb, nan_cb, hub_cb],  # nan guard before hub: a broken save is never pushed
    )
    print(f"precision: fp16={trainer.args.fp16} bf16={trainer.args.bf16}, "
          f"effective batch {trainer.args.per_device_train_batch_size * trainer.args.gradient_accumulation_steps}")
    print(f"GPU memory after model load: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# Cell 9 -- train. Nothing in this cell may stop the notebook from finishing: the Hub already
# holds every checkpoint, and a completed commit also keeps /kaggle/working as a second copy.
import traceback

outcome = "already finished" if ALREADY_FINISHED else "not started"
if not ALREADY_FINISHED:
    try:
        trainer.train(resume_from_checkpoint=str(resume_from) if resume_from else None)
        outcome = "ok"
    except torch.cuda.OutOfMemoryError:
        outcome = "OUT OF MEMORY"
        traceback.print_exc()
    except Exception:
        outcome = "CRASHED"
        traceback.print_exc()
    print(f"\ntraining outcome: {outcome}")
    print(f"peak GPU memory: allocated {torch.cuda.max_memory_allocated() / 1e9:.1f} GB, "
          f"reserved {torch.cuda.max_memory_reserved() / 1e9:.1f} GB "
          f"of {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 10 -- finish up and say exactly what to do next.
import math

if ALREADY_FINISHED:
    print("Nothing to do: the Hub says this run is finished. final/ holds the adapter.")
else:
    step, total = trainer.state.global_step, trainer.state.max_steps
    finished = outcome == "ok" and step >= total and not nan_cb.tripped
    if finished:
        final_dir = OUTPUT_DIR / "final"
        trainer.save_model(str(final_dir))
        sync.push_final(final_dir, step, total)
        print(f"pushed final adapter at step {step}")

    print("\nrecent log:")
    for row in [h for h in trainer.state.log_history if "loss" in h or "eval_loss" in h][-8:]:
        print("  ", {k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})

    steps_now = step - START_STEP
    hours_now = (time.time() - SESSION_START) / 3600
    print("\n" + "=" * 70)
    print(f"session: steps {START_STEP} -> {step} of {total} ({steps_now} steps, {hours_now:.1f} h incl. setup)")
    print(f"outcome: {outcome}; last checkpoint on the Hub: step {hub_cb.last_pushed_step}; "
          f"stop reason: {clock_cb.stop_reason or 'schedule complete'}")
    if nan_cb.tripped:
        print("!! NaN guard tripped: loss went non-finite. Do NOT just rerun -- look at the log first.")
    elif outcome != "ok":
        print("!! Training did not end cleanly (see traceback above). The Hub keeps the last good checkpoint.")
    elif RUN_MODE == "smoke" and STOP_AFTER_STEP:
        print("SMOKE PART 1 OK -> commit again with RUN_MODE='smoke'; it must say 'resuming from step 3'.")
    elif RUN_MODE == "smoke":
        print("SMOKE PASSED -> set RUN_MODE='train' and commit. You can close the laptop.")
    elif finished:
        print("TRAINING FINISHED. Adapter at final/ and adapters/step-* on the Hub. Next: evaluation.")
    else:
        rate = (time.time() - SESSION_START) / 3600 / max(steps_now, 1)
        print(f"Not done yet. Commit again (same settings) -- roughly "
              f"{math.ceil((total - step) * rate / BUDGET_HOURS)} more session(s).")
    print("=" * 70)